# Lab 7: AgentCore Memory — From Amnesia to Institutional Knowledge
## NovaPay · Killing the Repeat False Positive

**Level path:** 100 (concepts) → 200 (short-term memory) → 300 (long-term strategies) → 400 (production patterns)

**Duration:** ~75 minutes
**Prerequisites:** Labs 1–6
**Cost:** ~5 cents. Memory is metered per event/record/retrieval — there is no idle fee
**Region:** us-west-2 (AgentCore Memory GA region — change `REGION` below if yours differs)

---
# LEVEL 100 · The Business Problem

## The customer situation

Amara Okafor runs an import business in Lagos. Every quarter she travels to Accra for three
days to meet suppliers. Every quarter, the same thing happens:

1. She taps her NovaPay card in Accra.
2. The fraud engine flags it: **unusual location + new merchant + above-median amount**.
3. Her card is frozen mid-transaction, in a shop, in front of a supplier.
4. She calls support. Waits. Explains — *again* — that she travels to Accra quarterly.
5. An agent unblocks it and closes the ticket.

Next quarter, step 1 happens again. **Nobody remembers.**

## Why this is a real business problem, not an annoyance

NovaPay's own numbers for a representative month:

| Metric | Value | What it means |
|---|---|---|
| Flagged transactions / month | 4,200 | The fraud engine is doing its job |
| False positives | 1,890 (45%) | Nearly half of all freezes are wrong |
| **Repeat** false positives | **340 (18% of FPs)** | Same customer, same pattern, already resolved once |
| Handling time per repeat FP | ~8 min | Agent re-derives context that already existed |
| Churn after 2+ FPs in a quarter | 12% | The single strongest churn predictor in their data |

The 340 repeat false positives are the interesting number. Those are not a model-accuracy
problem. **The information needed to prevent them already existed** — a human resolved the
identical case last quarter and wrote it in a ticket. It simply was not available to the
system at the moment of the decision.

## Why your Labs 1–6 agent does not fix this

The agent you built has tools, guardrails, orchestration, retrieval and a deployment target.
It also has total amnesia. Lab 2 gave it a sliding window *inside a conversation* — when the
session ends, it is gone. Every call starts from zero.

Retrieval (Lab 5) does not solve it either: RAG retrieves from *documents you authored*.
Nothing in this loop ever **writes back** what was learned about this specific customer.

> **The gap:** your agent can look things up. It cannot remember.

## Where this lab takes you — A to B

| | **A — where you are now** | **B — where you will be** |
|---|---|---|
| New session | Agent knows nothing about Amara | Agent recalls her travel pattern and prior resolutions |
| Accra transaction | Flagged, frozen, customer calls | Recognised as established behaviour, risk adjusted down |
| Customer effort | Re-explains every quarter | Explains once, ever |
| Resolution | ~8 min, human required | Seconds, autonomous, with cited evidence |
| Institutional knowledge | Lives in closed tickets nobody reads | Queryable by the agent at decision time |

By the end you will have measured this — the same transaction scored twice, once without
memory and once with, with the difference explained by retrieved evidence.

## The seven things you will do

1. Create an AgentCore Memory resource (short-term)
2. Write conversation events into it and read them back within a session
3. Prove the amnesia problem — start a new session and watch the context vanish
4. Add long-term strategies (semantic, user-preference, summary) with namespace design
5. Backfill three historical NovaPay sessions and let extraction run
6. Query extracted memory semantically and wire it into a fraud-scoring tool
7. Run the A/B: same transaction, agent without memory vs agent with memory

In [ ]:
%pip install -q bedrock-agentcore strands-agents strands-agents-tools "boto3>=1.39.0"

In [ ]:
import json
import time
import uuid
from datetime import datetime, timedelta, timezone

import boto3
from botocore.config import Config as BotocoreConfig
from botocore.exceptions import ClientError
from IPython.display import HTML, display

# ── Configuration ────────────────────────────────────────────────────
REGION = "us-west-2"          # AgentCore Memory region
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Unique suffix so repeated runs never collide
RUN_ID = uuid.uuid4().hex[:8]
MEMORY_NAME = f"NovaPayFraudMemory_{RUN_ID}"

# NovaPay identifiers used throughout
ACTOR_AMARA = "CUST-1001"     # Amara Okafor
ACTOR_KWAME = "CUST-1002"     # Kwame Mensah (used for the isolation proof)

sts = boto3.client("sts", region_name=REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]

print(f"Account : {ACCOUNT_ID}")
print(f"Region  : {REGION}")
print(f"Run ID  : {RUN_ID}")
print(f"Memory  : {MEMORY_NAME}")

---
## 100.1 · What "memory" actually means in AgentCore

Two different things share the word. Keeping them separate is most of the battle.

```
                        ┌──────────────────────────────────────┐
   raw conversation     │       SHORT-TERM  (events)           │
   turns, verbatim  ──▶ │  every message, in order, per        │
                        │  (actor, session). Immediate recall. │
                        └───────────────┬──────────────────────┘
                                        │  background extraction
                                        ▼
                        ┌──────────────────────────────────────┐
                        │       LONG-TERM  (strategies)        │
                        │  distilled insight, across sessions: │
                        │   • semantic facts                    │
                        │   • user preferences                  │
                        │   • session summaries                 │
                        │   • episodes                          │
                        └──────────────────────────────────────┘
```

| | Short-term | Long-term |
|---|---|---|
| Unit | Event (a turn) | Extracted record |
| Scope | One session | Across all sessions for an actor |
| Written by | You, explicitly | AgentCore, asynchronously |
| Read by | `get_last_k_turns` / `list_events` | `retrieve_memories` (semantic query) |
| Latency to availability | Immediate | Seconds to a couple of minutes |
| Answers | "What did they just say?" | "What do we know about this person?" |

**The distinction that matters for NovaPay:** short-term memory means Amara does not have to
repeat herself *inside* the call. Long-term memory means she does not have to repeat herself
*next quarter*. The 340 repeat false positives are entirely a long-term memory problem.

### The four built-in strategies

| Strategy | Extracts | NovaPay use |
|---|---|---|
| `semanticMemoryStrategy` | Facts and entities | "TXN-102 was reviewed and cleared as legitimate travel" |
| `userPreferenceMemoryStrategy` | Preferences and patterns | "Travels to Accra quarterly for business" |
| `summaryMemoryStrategy` | Running session summaries | "Customer disputed a freeze; resolved as false positive" |
| `episodicMemoryStrategy` | Structured episodes + reflections | "Pattern: travel-flag disputes resolve as FP 90% of the time" |

This lab uses the first three. Episodic is the natural extension once you have this working.

---
# LEVEL 200 · Short-Term Memory — Continuity Inside a Session

Start with the simplest possible memory resource: no strategies, just an event log.
This is the layer that gives an agent conversational continuity.

> ### ⚠️ First production gotcha — resource lifecycle
> `create_memory` returns as soon as AWS *accepts* the request. The resource is then in
> status `CREATING`, not `ACTIVE`. Write an event too early and you get:
>
> ```
> ValidationException: Memory status is not active, unable to process CreateEvent request
> ```
>
> This is the standard AWS control-plane pattern — creation is asynchronous — and it bites
> everyone once. The cell below uses `create_memory_and_wait`, and defines an explicit
> `wait_for_memory_active` poller as a fallback. **In your own code, never assume a freshly
> created AgentCore resource is immediately usable.**

In [ ]:
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=REGION)


def memory_status(memory_id):
    """Current status of a memory resource, tolerating SDK surface differences."""
    for getter, kwargs in (
        (getattr(memory_client, "get_memory", None), {"memory_id": memory_id}),
        (getattr(memory_client, "gmcp_client", None), None),
    ):
        if getter is None:
            continue
        try:
            if kwargs is not None:
                r = getter(**kwargs)
            else:
                r = getter.get_memory(memoryId=memory_id)
            m = r.get("memory", r)
            return m.get("status")
        except Exception:
            continue
    return None


def wait_for_memory_active(memory_id, timeout=300, interval=10):
    """Block until the memory resource is ACTIVE.

    create_memory returns as soon as the resource is accepted — status CREATING.
    Calling create_event before it reaches ACTIVE raises:
        ValidationException: Memory status is not active
    """
    started = time.time()
    last = None
    while time.time() - started < timeout:
        last = memory_status(memory_id)
        if last == "ACTIVE":
            print(f"   Status: ACTIVE (after ~{int(time.time() - started)}s)")
            return True
        if last in ("FAILED", "DELETING"):
            raise RuntimeError(f"Memory entered terminal status: {last}")
        print(f"   Status: {last or 'CREATING'} — waiting {interval}s…")
        time.sleep(interval)
    raise TimeoutError(f"Memory {memory_id} not ACTIVE after {timeout}s (last: {last})")


# A memory resource with NO strategies = short-term only (raw event storage).
# Use the _and_wait variant so the resource is ACTIVE before we write events.
try:
    st_memory = memory_client.create_memory_and_wait(
        name=f"{MEMORY_NAME}_ST",
        description="NovaPay short-term conversation memory (Level 200)",
        strategies=[],
    )
    ST_MEMORY_ID = st_memory.get("id") or st_memory.get("memoryId")
    print(f"✅ Short-term memory ACTIVE: {ST_MEMORY_ID}")
except (AttributeError, TypeError):
    # Older SDKs have no create_memory_and_wait — create then poll ourselves.
    st_memory = memory_client.create_memory(
        name=f"{MEMORY_NAME}_ST",
        description="NovaPay short-term conversation memory (Level 200)",
    )
    ST_MEMORY_ID = st_memory.get("id") or st_memory.get("memoryId")
    print(f"✅ Short-term memory created: {ST_MEMORY_ID}")
    wait_for_memory_active(ST_MEMORY_ID)

## 200.1 · Write a real support conversation into memory

`create_event` takes a list of `(text, role)` tuples. Roles are `USER`, `ASSISTANT`, `TOOL`.

Note the `TOOL` entries — recording tool calls, not just chat, is what later lets the agent
(and an auditor) reconstruct *why* a decision was made. Skipping them is the most common
mistake teams make here.

In [ ]:
SESSION_Q3 = f"accra-freeze-q3-{RUN_ID}"

memory_client.create_event(
    memory_id=ST_MEMORY_ID,
    actor_id=ACTOR_AMARA,
    session_id=SESSION_Q3,
    messages=[
        ("My card just got declined at a shop in Accra. I'm standing here with my supplier.",
         "USER"),
        ("I'm sorry about that. Let me look at the transaction right away.",
         "ASSISTANT"),
        ("fraud_check(transaction_id='TXN-4471')", "TOOL"),
        ("get_customer_profile(customer_id='CUST-1001')", "TOOL"),
        ("I can see it — TXN-4471, 89,000 NGN in Accra, flagged for unusual location and a "
         "new merchant. Can you confirm you're travelling?",
         "ASSISTANT"),
        ("Yes. I go to Accra every quarter for supplier meetings. Same trip, same shops. "
         "This happens to me every single time.",
         "USER"),
        ("Understood — I've cleared the block and TXN-4471 is approved.",
         "ASSISTANT"),
        ("clear_fraud_hold(transaction_id='TXN-4471', reason='verified_business_travel')",
         "TOOL"),
        ("Thank you. Please make a note so it stops happening.", "USER"),
        ("Noted: quarterly business travel to Accra, supplier meetings, recurring merchants.",
         "ASSISTANT"),
    ],
)

print(f"✅ Event written to session: {SESSION_Q3}")

## 200.2 · Read it back — continuity within the session

In [ ]:
def load_recent_turns(memory_id: str, actor_id: str, session_id: str, k: int = 10):
    """Read recent turns. Falls back across SDK surface differences."""
    try:
        return memory_client.get_last_k_turns(
            memory_id=memory_id, actor_id=actor_id, session_id=session_id, k=k
        )
    except (AttributeError, ClientError, TypeError):
        try:
            return memory_client.list_events(
                memory_id=memory_id, actor_id=actor_id, session_id=session_id, max_results=k
            )
        except Exception as e:
            print(f"⚠️  Could not list events: {e}")
            return []


turns = load_recent_turns(ST_MEMORY_ID, ACTOR_AMARA, SESSION_Q3, k=10)
print(f"Recovered {len(turns)} turn group(s) from short-term memory\n")
print(json.dumps(turns, indent=2, default=str)[:1800])

## 200.3 · Now prove the problem

Everything above works. The agent has perfect recall — *of this session*.

Amara's next call is a **new session**. Same customer, same actor ID, three months later.
Watch what the agent has to work with.

In [ ]:
SESSION_Q4 = f"accra-freeze-q4-{RUN_ID}"   # New quarter, new session, same customer

q4_turns = load_recent_turns(ST_MEMORY_ID, ACTOR_AMARA, SESSION_Q4, k=10)

print("=" * 68)
print("NEW SESSION — what the agent knows about Amara")
print("=" * 68)
print(f"Turns available: {len(q4_turns)}")
print()
if not q4_turns:
    print("🚨 NOTHING.")
    print()
    print("   Same customer. Same actor_id. Same recurring situation.")
    print("   Short-term memory is scoped to a session — so on this call the agent")
    print("   will ask her to explain the Accra trip for the fourth time.")
    print()
    print("   This is the 340-repeat-false-positive problem, reproduced in one cell.")
else:
    print(q4_turns)

> ### Level 200 checkpoint
> You can persist and recall conversation turns. You have also reproduced the exact failure
> that costs NovaPay 340 repeat false positives a month. Short-term memory is necessary and
> **not sufficient** — session-scoped recall cannot survive the thing that matters here,
> which is the gap between quarters.

---
# LEVEL 300 · Long-Term Memory — Knowledge That Outlives the Session

Long-term memory works by **extraction**: you keep writing ordinary events, and AgentCore
runs background strategies that distil them into durable, queryable records.

## 300.1 · Namespace design — do this deliberately

A namespace is the address of an extracted memory. The template is interpolated with
`{actorId}`, `{sessionId}`, `{memoryStrategyId}` at write time, and you query by prefix.

**Namespace design is access-control design.** Get it right here and multi-tenant isolation
is structural. Get it wrong and one customer's history is retrievable under another's query.

| Strategy | Namespace template | Why this shape |
|---|---|---|
| User preference | `/novapay/customers/{actorId}/preferences/` | Per-customer, spans all sessions — travel patterns live here |
| Semantic facts | `/novapay/customers/{actorId}/facts/` | Per-customer, spans all sessions — resolved cases live here |
| Summary | `/novapay/customers/{actorId}/sessions/{sessionId}/summary/` | Per-session — one summary per contact |

Note that every template is rooted at `{actorId}`. That is the isolation boundary, and we
will prove it holds in Level 400.

In [ ]:
LT_STRATEGIES = [
        {
            "userPreferenceMemoryStrategy": {
                "name": "NovaPayCustomerPatterns",
                "namespaceTemplates": ["/novapay/customers/{actorId}/preferences/"],
            }
        },
        {
            "semanticMemoryStrategy": {
                "name": "NovaPayCaseFacts",
                "namespaceTemplates": ["/novapay/customers/{actorId}/facts/"],
            }
        },
        {
            "summaryMemoryStrategy": {
                "name": "NovaPayContactSummary",
                "namespaceTemplates": [
                    "/novapay/customers/{actorId}/sessions/{sessionId}/summary/"
                ],
            }
        },
]

LT_DESCRIPTION = "NovaPay long-term customer memory: travel patterns, resolved fraud cases"

try:
    lt_memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME, description=LT_DESCRIPTION, strategies=LT_STRATEGIES
    )
except (AttributeError, TypeError):
    lt_memory = memory_client.create_memory(
        name=MEMORY_NAME, description=LT_DESCRIPTION, strategies=LT_STRATEGIES
    )

MEMORY_ID = lt_memory.get("id") or lt_memory.get("memoryId")

# Belt and braces: strategy provisioning can lag the create call.
if memory_status(MEMORY_ID) != "ACTIVE":
    wait_for_memory_active(MEMORY_ID)

print(f"✅ Long-term memory ACTIVE: {MEMORY_ID}")
print(f"   Strategies: {len(lt_memory.get('strategies', lt_memory.get('memoryStrategies', [])))}")

## 300.2 · Backfill the history that already existed

This is the step that mirrors what a real engagement looks like. NovaPay *has* this data —
it is sitting in closed support tickets. We replay three prior contacts so extraction has
something to work with.

In production this backfill is a one-off migration job over your ticket history, and it is
usually where the majority of the value is unlocked on day one.

In [ ]:
HISTORICAL_SESSIONS = [
    {
        "session_id": f"accra-freeze-q1-{RUN_ID}",
        "messages": [
            ("My card was blocked in Accra again. Transaction TXN-2210.", "USER"),
            ("fraud_check(transaction_id='TXN-2210')", "TOOL"),
            ("It was flagged for unusual location. Are you travelling?", "ASSISTANT"),
            ("Yes, I'm in Accra for my quarterly supplier meetings. I come every quarter.",
             "USER"),
            ("clear_fraud_hold(transaction_id='TXN-2210', reason='verified_business_travel')",
             "TOOL"),
            ("Cleared. TXN-2210 is approved — verified business travel to Accra.",
             "ASSISTANT"),
        ],
    },
    {
        "session_id": f"accra-freeze-q2-{RUN_ID}",
        "messages": [
            ("Blocked in Accra again, TXN-3105. This is the second time this year.", "USER"),
            ("fraud_check(transaction_id='TXN-3105')", "TOOL"),
            ("I see the flag — unusual location, Accra. Same quarterly trip?", "ASSISTANT"),
            ("Yes. Same trip, same suppliers. Amounts are usually 50,000 to 95,000 NGN.",
             "USER"),
            ("clear_fraud_hold(transaction_id='TXN-3105', reason='verified_business_travel')",
             "TOOL"),
            ("Cleared as a false positive. I've recorded the travel pattern.", "ASSISTANT"),
        ],
    },
    {
        "session_id": f"profile-update-{RUN_ID}",
        "messages": [
            ("I want to set expectations: I travel to Accra in the first week of every "
             "quarter, and to Abidjan once a year in November.", "USER"),
            ("Noted. Quarterly Accra travel, annual Abidjan travel in November.",
             "ASSISTANT"),
            ("Also I prefer SMS alerts, not calls. And never freeze without messaging me "
             "first — call me only if I don't reply within ten minutes.", "USER"),
            ("Recorded: SMS-first contact preference, 10-minute reply window before calling.",
             "ASSISTANT"),
        ],
    },
]

for s in HISTORICAL_SESSIONS:
    memory_client.create_event(
        memory_id=MEMORY_ID,
        actor_id=ACTOR_AMARA,
        session_id=s["session_id"],
        messages=s["messages"],
    )
    print(f"  ✅ ingested {s['session_id']}  ({len(s['messages'])} turns)")

# Write one session for a DIFFERENT customer — used for the isolation proof at Level 400
memory_client.create_event(
    memory_id=MEMORY_ID,
    actor_id=ACTOR_KWAME,
    session_id=f"kwame-utilities-{RUN_ID}",
    messages=[
        ("Did my utility payment TXN-103 to Accra Utilities go through?", "USER"),
        ("get_transaction(transaction_id='TXN-103')", "TOOL"),
        ("Yes — 150 GHS to Accra Utilities, completed.", "ASSISTANT"),
        ("Good. I only ever pay utilities and rent through NovaPay, nothing else.", "USER"),
        ("Noted: utilities and rent payments only.", "ASSISTANT"),
    ],
)
print(f"  ✅ ingested Kwame's session (for isolation testing)")
print(f"\n📥 {len(HISTORICAL_SESSIONS)} Amara sessions + 1 Kwame session written.")

## 300.3 · Wait for extraction

Extraction is asynchronous — this is a deliberate design choice by AWS, so writing events
never blocks your request path. The cost is that insight is not instantly queryable.

Typical: 30–90 seconds. The cell below polls rather than sleeping blindly, so it returns as
soon as records appear.

In [ ]:
def wait_for_extraction(memory_id, namespace, query, timeout=240, interval=15):
    """Poll retrieve_memories until extraction produces records (or timeout)."""
    started = time.time()
    attempt = 0
    while time.time() - started < timeout:
        attempt += 1
        try:
            found = memory_client.retrieve_memories(
                memory_id=memory_id, namespace=namespace, query=query
            )
            if found:
                elapsed = int(time.time() - started)
                print(f"✅ Extraction complete after ~{elapsed}s ({len(found)} record(s))")
                return found
        except ClientError as e:
            print(f"   attempt {attempt}: {e.response['Error']['Code']}")
        except Exception as e:
            print(f"   attempt {attempt}: {type(e).__name__}")
        print(f"   attempt {attempt}: not ready, waiting {interval}s…")
        time.sleep(interval)

    print("⚠️  Timed out waiting for extraction.")
    print("    Long-term extraction can take longer on first use — re-run this cell.")
    return []


NS_PREFS = f"/novapay/customers/{ACTOR_AMARA}/preferences/"
NS_FACTS = f"/novapay/customers/{ACTOR_AMARA}/facts/"

print("⏳ Waiting for preference extraction…")
prefs = wait_for_extraction(MEMORY_ID, NS_PREFS, "travel patterns and contact preferences")

## 300.4 · Query extracted memory semantically

In [ ]:
def show_memories(records, title):
    print("=" * 68)
    print(title)
    print("=" * 68)
    if not records:
        print("   (none yet — extraction may still be running)")
        return
    for i, rec in enumerate(records, 1):
        content = rec.get("content", rec)
        if isinstance(content, dict):
            text = content.get("text") or json.dumps(content)
        else:
            text = str(content)
        print(f"\n  [{i}] {text.strip()[:400]}")
        if rec.get("memoryStrategyId"):
            print(f"      ↳ strategy: {rec['memoryStrategyId']}")
    print()


# The question the agent actually needs answered at decision time
travel = memory_client.retrieve_memories(
    memory_id=MEMORY_ID,
    namespace=NS_PREFS,
    query="Does this customer travel to Accra regularly?",
)
show_memories(travel, "PREFERENCES · 'Does this customer travel to Accra regularly?'")

facts = memory_client.retrieve_memories(
    memory_id=MEMORY_ID,
    namespace=NS_FACTS,
    query="previous fraud holds that were cleared as false positives",
)
show_memories(facts, "FACTS · 'previous fraud holds cleared as false positives'")

> ### Level 300 checkpoint
> The knowledge that was locked in closed tickets is now **queryable by semantic question**,
> scoped per customer, and survives across sessions. You did not write a summariser, an
> embedding pipeline, or a vector store — you declared three strategies and kept writing
> ordinary events.
>
> Next: turn this from a retrieval demo into a decision that changes.

---
# LEVEL 400 · Production Patterns

Four things separate a memory demo from a memory system you can put in front of a regulator.

1. **Isolation** — prove one customer cannot retrieve another's history
2. **Memory as a signal** — memory must change a *decision*, not just decorate a response
3. **Measurement** — prove the A→B with the same input scored both ways
4. **Governance** — PII, retention, and cost

## 400.1 · Prove tenant isolation

Namespaces are rooted at `{actorId}`. Query Amara's namespace with a question that only
Kwame's history can answer. If the design holds, you get nothing.

In [ ]:
NS_KWAME_PREFS = f"/novapay/customers/{ACTOR_KWAME}/preferences/"

cross = memory_client.retrieve_memories(
    memory_id=MEMORY_ID,
    namespace=NS_PREFS,                       # Amara's namespace
    query="utilities and rent payments only",  # a fact that exists only in Kwame's history
)

own = memory_client.retrieve_memories(
    memory_id=MEMORY_ID,
    namespace=NS_KWAME_PREFS,                 # Kwame's own namespace
    query="utilities and rent payments only",
)

print("=" * 68)
print("TENANT ISOLATION TEST")
print("=" * 68)
print(f"Kwame's fact queried in AMARA's namespace : {len(cross)} record(s)")
print(f"Kwame's fact queried in KWAME's namespace : {len(own)} record(s)")
print()
if len(cross) == 0:
    print("✅ PASS — no cross-tenant leakage.")
    print("   Isolation is structural: it comes from the namespace template,")
    print("   not from application code remembering to filter.")
else:
    print("🚨 FAIL — investigate namespace templates before going further.")
    show_memories(cross, "LEAKED RECORDS")

## 400.2 · Wire memory into the fraud decision

This is the step most teams skip, and it is the entire point. Memory that only makes the
chat friendlier does not move a business metric. Memory that **adjusts a risk score** does.

The tool below does what a production fraud service would: computes a base risk from the
signals, then queries memory for exculpatory history and applies a documented adjustment —
returning the evidence alongside the number so a human reviewer (or an auditor) can see
exactly why the score moved.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

# Live transaction under review — Amara's Q4 Accra trip
LIVE_TXN = {
    "transaction_id": "TXN-5588",
    "customer_id": ACTOR_AMARA,
    "amount": 91500.00,
    "currency": "NGN",
    "location": "Accra, GH",
    "merchant": "Kofi Textiles Ltd",
    "timestamp": "2024-10-02T11:15:00Z",
    "raw_flags": ["unusual_location", "new_merchant", "above_median_amount"],
}

FLAG_WEIGHTS = {"unusual_location": 35, "new_merchant": 25, "above_median_amount": 20}


def _memory_text(rec):
    content = rec.get("content", rec)
    if isinstance(content, dict):
        return (content.get("text") or json.dumps(content)).strip()
    return str(content).strip()


@tool
def assess_fraud_risk(transaction_id: str, use_memory: bool = True) -> str:
    """Assess fraud risk for a NovaPay transaction, optionally using customer memory.

    Args:
        transaction_id: Transaction to assess (e.g. TXN-5588)
        use_memory: Whether to consult long-term customer memory for mitigating history
    """
    txn = LIVE_TXN if transaction_id == LIVE_TXN["transaction_id"] else None
    if not txn:
        return json.dumps({"error": f"Transaction {transaction_id} not found"})

    base_score = sum(FLAG_WEIGHTS.get(f, 10) for f in txn["raw_flags"])

    result = {
        "transaction_id": transaction_id,
        "base_risk_score": base_score,
        "raw_flags": txn["raw_flags"],
        "memory_consulted": use_memory,
        "mitigating_evidence": [],
        "adjustment": 0,
        "final_risk_score": base_score,
    }

    if use_memory:
        evidence = []
        for ns, q in [
            (f"/novapay/customers/{txn['customer_id']}/preferences/",
             f"does this customer travel to {txn['location'].split(',')[0]} regularly"),
            (f"/novapay/customers/{txn['customer_id']}/facts/",
             "previous fraud holds cleared as verified business travel"),
        ]:
            try:
                for rec in memory_client.retrieve_memories(
                    memory_id=MEMORY_ID, namespace=ns, query=q
                ):
                    evidence.append(_memory_text(rec)[:300])
            except Exception as e:
                result.setdefault("memory_errors", []).append(str(e))

        # Deduplicate while preserving order
        seen, deduped = set(), []
        for e in evidence:
            if e not in seen:
                seen.add(e)
                deduped.append(e)
        evidence = deduped

        blob = " ".join(evidence).lower()
        adjustment = 0
        if "accra" in blob and ("travel" in blob or "quarter" in blob):
            adjustment -= 35          # location is established behaviour, not anomalous
        if "cleared" in blob or "false positive" in blob or "verified" in blob:
            adjustment -= 20          # this exact pattern was human-reviewed before

        result["mitigating_evidence"] = evidence
        result["adjustment"] = adjustment
        result["final_risk_score"] = max(0, base_score + adjustment)

    score = result["final_risk_score"]
    result["risk_level"] = "high" if score >= 60 else "medium" if score >= 35 else "low"
    result["recommended_action"] = (
        "freeze_and_contact" if score >= 60
        else "monitor" if score >= 35
        else "approve"
    )
    return json.dumps(result, indent=2)


print("🔧 assess_fraud_risk tool defined")
print(f"   Base score for {LIVE_TXN['transaction_id']}: "
      f"{sum(FLAG_WEIGHTS.get(f, 10) for f in LIVE_TXN['raw_flags'])} "
      f"(flags: {', '.join(LIVE_TXN['raw_flags'])})")

## 400.3 · The A/B — the whole lab in one output

Same transaction. Same tool. Same model. The only variable is whether memory is consulted.

In [ ]:
print("=" * 68)
print("A · WITHOUT MEMORY  (your Labs 1–6 agent)")
print("=" * 68)
without = json.loads(assess_fraud_risk(LIVE_TXN["transaction_id"], use_memory=False))
print(json.dumps(without, indent=2))

print()
print("=" * 68)
print("B · WITH MEMORY  (this lab)")
print("=" * 68)
with_mem = json.loads(assess_fraud_risk(LIVE_TXN["transaction_id"], use_memory=True))
print(json.dumps(with_mem, indent=2))

print()
print("=" * 68)
print("DELTA")
print("=" * 68)
print(f"  Risk score   : {without['final_risk_score']}  →  {with_mem['final_risk_score']}")
print(f"  Risk level   : {without['risk_level']}  →  {with_mem['risk_level']}")
print(f"  Action       : {without['recommended_action']}  →  {with_mem['recommended_action']}")
print(f"  Evidence used: {len(with_mem['mitigating_evidence'])} memory record(s)")
print()
if with_mem["recommended_action"] != without["recommended_action"]:
    print("  ✅ Memory changed the DECISION, not just the wording of the answer.")
    print("     Amara's card is not frozen. She never has to make the call.")
    print("     That is one of the 340 repeat false positives, eliminated.")
else:
    print("  ⚠️  No decision change — extraction may still be incomplete.")
    print("     Re-run cell 300.3 to finish extraction, then re-run this cell.")

## 400.4 · The agent with memory-aware tooling

Now put the tool behind an agent, so the reasoning and the citation are visible.

In [ ]:
bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    boto_client_config=BotocoreConfig(
        retries={"max_attempts": 3}, connect_timeout=5, read_timeout=90
    ),
)

fraud_agent = Agent(
    model=bedrock_model,
    tools=[assess_fraud_risk],
    system_prompt="""You are NovaPay's Fraud Triage Agent.

Assess flagged transactions using the assess_fraud_risk tool with use_memory=True.

In your answer you must:
1. State the final risk level and the recommended action.
2. Explicitly cite the mitigating evidence retrieved from customer memory.
3. Say how the memory changed the outcome versus the base score.

Be concise and factual. Never invent evidence that is not in the tool result.""",
)

print("=" * 68)
print("FRAUD TRIAGE AGENT — with long-term customer memory")
print("=" * 68)
response = fraud_agent(
    f"Transaction {LIVE_TXN['transaction_id']} has been flagged. "
    "Assess it and tell me whether to freeze the card."
)
print(f"\n🤖 {response}")

## 400.5 · Governance — the three questions you will be asked

### PII inside memory
Extraction writes model-generated text into durable storage. If a customer says their SSN
out loud, a naive semantic strategy can persist it. Two controls, used together:

- **Lab 3 guardrails on the way in** — PII is masked before the event is ever written
- **Namespace scoping** — retrieval is bounded to the actor, so blast radius is one customer

Memory does not exempt you from data protection. It *raises* the stakes, because it is the
one component whose whole job is to remember.

### Retention
Memory resources support expiry. In a regulated business your retention window is a legal
input, not an engineering preference — align it with your data retention policy and be able
to show the configuration.

### Cost
You pay for storage plus extraction. Extraction is the variable cost, and it scales with
conversation volume, not customer count. Two levers: only enable strategies you actually
query, and do not write low-value events (heartbeats, retries) into memory at all.

## 400.6 · What you would do next in a real engagement

| Next step | Why |
|---|---|
| Backfill 12 months of closed tickets | The 340/month number is historical — the value is available immediately |
| Add `episodicMemoryStrategy` | Learns *resolution patterns* across customers, not just facts about one |
| Feed adjustments back to the fraud model | Memory becomes labelled training data for the scorer itself |
| Alarm on adjustment magnitude | A sudden spike in downward adjustments is either a great win or an attack |
| Human review on large adjustments | Keep a person in the loop where the score moves most |

> ### ⚠️ The failure mode worth naming
> Memory can be poisoned. If an attacker can get "this customer always travels to Accra"
> written into memory, they have manufactured their own exculpatory evidence. Mitigations:
> only extract from **authenticated** sessions, treat memory as a signal that *lowers*
> confidence thresholds rather than one that auto-approves, and alarm when a single actor's
> preference set changes rapidly. This is the question a principal architect is expected to
> raise unprompted.

## 400.7 · Teardown — always run this

In [ ]:
def safe_delete_memory(memory_id, label):
    if not memory_id:
        return
    try:
        memory_client.delete_memory(memory_id=memory_id)
        print(f"🗑️  Deleted {label}: {memory_id}")
    except Exception as e:
        print(f"⚠️  Could not delete {label} ({memory_id}): {e}")


safe_delete_memory(globals().get("ST_MEMORY_ID"), "short-term memory")
safe_delete_memory(globals().get("MEMORY_ID"), "long-term memory")

print("\n✅ Teardown complete.")
print("   Verify in the console: Bedrock AgentCore → Memory")

---
# Knowledge Check

1. Why does short-term memory not solve NovaPay's repeat false positive problem?
2. What is the operational consequence of extraction being asynchronous?
3. You are asked to prove one customer cannot read another's memory. What do you show?
4. Why is wiring memory into the *risk score* more valuable than into the response text?
5. A customer's memory says "always travels to Accra." How do you know that is trustworthy?
6. Which strategy would you add to learn resolution patterns across many customers, and why?

### Answers

<details>
<summary>Click to reveal</summary>

1. Short-term memory is scoped to a `session_id`. The repeat false positive happens across a
   three-month gap — a different session entirely. Session-scoped recall structurally cannot
   span it. You need extraction into an actor-scoped namespace that outlives the session.

2. You cannot write an event and immediately query the insight derived from it. Any design
   that assumes read-after-write on long-term memory will be flaky. Architect for it: write
   events on the request path, consult memory on a subsequent interaction, and never block a
   customer-facing call waiting for extraction.

3. The namespace templates — every one rooted at `{actorId}` — plus the executed isolation
   test from 400.1 showing a query in Amara's namespace returning zero records for a fact
   that exists only in Kwame's history. The point is that isolation is structural, not a
   filter the application code has to remember to apply.

4. Because a risk score drives an action (freeze / monitor / approve) and an action is what
   the business measures. Memory in the response text makes the agent sound informed; memory
   in the score means the card is not frozen. The A/B in 400.3 shows the decision flipping
   from `freeze_and_contact` to `approve` — that is the business outcome.

5. You do not, by default — that is the memory poisoning risk. It is trustworthy only if it
   was extracted from an authenticated session, and even then you should use it to lower a
   threshold rather than to auto-approve, alarm on rapid preference changes for a single
   actor, and keep human review on large downward adjustments.

6. `episodicMemoryStrategy`. Semantic and preference strategies build a profile of *one*
   customer. Episodic captures structured episodes — scenario, action, outcome — and reflects
   across them, so it can learn "travel-flag disputes resolve as false positives ~90% of the
   time," which is a policy-level insight rather than a per-customer fact.

</details>

---

## What you can now do

You can take a customer who says *"our agent has no continuity and our people keep
re-deriving context that already exists"* and:

- separate the short-term and long-term problem, and say which one their symptom actually is
- design namespaces so tenant isolation is structural and provable
- choose strategies against the questions the business needs answered
- plan the historical backfill that delivers value on day one
- wire memory into a **decision** and measure the delta
- name the poisoning, PII, retention and cost risks before the customer's security team does

*NovaPay AI Agent Training — Lab 7 Complete*